# Waste-Aware Calibrated Early-Exit Experiment

This notebook is the only experiment entry point. It audits the data, trains every comparison model, calibrates each exit, searches class-aware thresholds, freezes the chosen settings, evaluates the locked test set, and shows every main table and chart inline.

The default smoke run checks the software pipeline. It is not final research evidence.

## 1. Setup and run mode

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Image as NotebookImage, Markdown, display

from waste_early_exit.conclusion import assess_conclusion
from waste_early_exit.config import load_config
from waste_early_exit.experiments import (
    aggregate_seed_comparisons,
    aggregate_seed_tables,
    build_seed_runners,
    run_seed_experiment,
)
from waste_early_exit.routing import route_cached_logits
from waste_early_exit.visualization import (
    plot_pareto,
    plot_qualitative_examples,
    plot_reliability_diagram,
    plot_threshold_heatmap,
    plot_training_history,
)

RUN_MODE = "smoke"  # change to "full" only after smoke passes
FORCE_REBUILD = False
PROPOSED_VARIANT = "self_distill"
PROJECT_ROOT = Path.cwd().resolve()
CONFIG_PATH = PROJECT_ROOT / "configs" / f"{RUN_MODE}.yaml"

def show_table(name, table, maximum_rows=30):
    display(Markdown(f"### {name}"))
    display(table.head(maximum_rows))
    if len(table) > maximum_rows:
        display(Markdown(f"Showing {maximum_rows:,} of {len(table):,} rows. The complete table is saved under `artifacts/`."))

def show_stage(stage, show_figures=True):
    display(Markdown(f"**{stage.message}**"))
    display(Markdown(f"Validity: `{stage.validity}`"))
    for name, table in stage.tables.items():
        show_table(name.replace("_", " ").title(), table)
    if show_figures:
        for name, path in stage.figures.items():
            display(Markdown(f"### {name.replace('_', ' ').title()}"))
            display(NotebookImage(filename=str(path)))

config = load_config(CONFIG_PATH, project_root=PROJECT_ROOT)
runners = build_seed_runners(config, project_root=PROJECT_ROOT, progress_display=True)
primary_seed = next(iter(runners))
runner = runners[primary_seed]
progress = runner.progress
primary_artifact_root = runner.artifact_root
display(Markdown(f"Overall log: `{progress.aggregate_log_path}`"))
display(Markdown(f"Seed log: `{progress.seed_log_paths[primary_seed]}`"))
setup_stage = runner.setup()
show_stage(setup_stage)

## 2. What this run can prove

A smoke run uses capped samples and short training. It proves that data checks, models, calibration, routing, profiling, tables, and plots work together. Only a completed full run with three seeds can support the final report.

In [ ]:
if RUN_MODE == "smoke":
    display(Markdown("# Pipeline validation only"))
    display(Markdown("Do not quote smoke accuracy, savings, latency, or energy as the final experiment result."))
else:
    display(Markdown("# Full experiment"))
    display(Markdown("Check all three seeds and confidence intervals before writing a claim."))

## 3. Audit the dataset

In [ ]:
audit_stage = runner.audit_data(force=FORCE_REBUILD)
show_stage(audit_stage, show_figures=False)

## 4. Freeze leakage-safe splits

In [ ]:
split_stage = runner.prepare_splits(force=FORCE_REBUILD)
show_stage(split_stage, show_figures=False)

## 5. Explore the data

In [ ]:
eda_stage = runner.run_eda()
show_stage(eda_stage)

## 6. Train the static baselines

In [ ]:
static_stage = runner.train_static_baselines()
show_stage(static_stage, show_figures=False)

for model_name, history in static_stage.tables["history"].groupby("model"):
    output_path = primary_artifact_root / "figures" / "training" / f"static_{model_name}.png"
    figure = plot_training_history(history, output_path)
    display(Markdown(f"### {model_name.replace('_', ' ').title()} training"))
    display(figure)
    plt.close(figure)

## 7. Train the early-exit models

In [ ]:
early_stage = runner.train_early_exit_models()
show_stage(early_stage, show_figures=False)

for variant, history in early_stage.tables["history"].groupby("variant"):
    output_path = primary_artifact_root / "figures" / "training" / f"early_{variant}.png"
    figure = plot_training_history(history, output_path)
    display(Markdown(f"### {variant.replace('_', ' ').title()} training"))
    display(figure)
    plt.close(figure)

## 8. Calibrate every exit

In [ ]:
calibration_stage = runner.calibrate_variant(PROPOSED_VARIANT)
show_stage(calibration_stage, show_figures=False)

model_key = f"early:{PROPOSED_VARIANT}"
calibration_bundle = runner.predictions[f"{model_key}:calibration"]
fitted = runner.calibrations[PROPOSED_VARIANT]
for exit_name in ("exit1", "exit2", "final"):
    raw_path = primary_artifact_root / "figures" / "calibration" / f"{exit_name}_raw.png"
    calibrated_path = primary_artifact_root / "figures" / "calibration" / f"{exit_name}_calibrated.png"
    raw_figure = plot_reliability_diagram(
        calibration_bundle.logits[exit_name],
        calibration_bundle.labels,
        raw_path,
        title=f"{exit_name.title()} before calibration",
    )
    calibrated_figure = plot_reliability_diagram(
        calibration_bundle.logits[exit_name] / fitted.temperatures[exit_name],
        calibration_bundle.labels,
        calibrated_path,
        title=f"{exit_name.title()} after calibration",
    )
    display(raw_figure)
    display(calibrated_figure)
    plt.close(raw_figure)
    plt.close(calibrated_figure)

## 9. Estimate class difficulty and search thresholds

In [ ]:
search_stage = runner.search_variant(PROPOSED_VARIANT)
show_stage(search_stage, show_figures=False)

threshold_path = primary_artifact_root / "figures" / "routing" / "class_thresholds.png"
threshold_figure = plot_threshold_heatmap(search_stage.tables["selected_thresholds"], threshold_path)
display(threshold_figure)
plt.close(threshold_figure)

search_plot_table = search_stage.tables["search"].copy()
search_plot_table["method"] = search_plot_table.apply(
    lambda row: f"t1={row.tau_exit1:.2f}, t2={row.tau_exit2:.2f}, lambda={row['lambda']:.2f}", axis=1
)
search_plot_table = search_plot_table.loc[search_plot_table["feasible"]].head(20)
search_pareto_path = primary_artifact_root / "figures" / "routing" / "threshold_pareto.png"
search_pareto = plot_pareto(search_plot_table, "average_flops", "macro_f1", search_pareto_path)
display(search_pareto)
plt.close(search_pareto)

## 10. Freeze the proposed settings

In [ ]:
freeze_stage = runner.freeze_variant(PROPOSED_VARIANT)
show_stage(freeze_stage, show_figures=False)
display(Markdown(f"Frozen manifest: `{freeze_stage.artifacts['frozen_manifest']}`"))

## 11. Evaluate the locked test set once

In [ ]:
locked_stage = runner.evaluate_locked_test(PROPOSED_VARIANT)
show_stage(locked_stage)

## 12. Run the ablations

In [ ]:
ablation_stage = runner.run_ablations(PROPOSED_VARIANT)
show_stage(ablation_stage, show_figures=False)

ablation_plot_path = primary_artifact_root / "figures" / "results" / "ablation_pareto.png"
ablation_figure = plot_pareto(
    ablation_stage.tables["ablations"],
    "average_flops",
    "macro_f1",
    ablation_plot_path,
)
display(ablation_figure)
plt.close(ablation_figure)

## 13. Profile FLOPs, latency, energy, and CO2e

In [ ]:
profiling_stage = runner.profile_variant(PROPOSED_VARIANT)
show_stage(profiling_stage, show_figures=False)

energy_status = profiling_stage.tables["energy"].loc[0, "status"]
if energy_status == "unavailable":
    display(Markdown("Energy is unavailable for this run. FLOPs and latency remain valid, but they do not prove an energy saving."))

## 14. Inspect early-exit successes and failures

In [ ]:
test_bundle = runner.predictions[f"{model_key}:test"]
proposed_route = route_cached_logits(
    test_bundle.logits,
    runner.calibrations[PROPOSED_VARIANT].temperatures,
    runner.searches[PROPOSED_VARIANT].thresholds,
)
test_frame = runner.mode_df.loc[runner.mode_df["split"] == "test"]
examples_path = primary_artifact_root / "figures" / "results" / "qualitative_examples.png"
examples_figure = plot_qualitative_examples(
    test_frame,
    test_bundle.sample_ids,
    test_bundle.labels.numpy(),
    proposed_route.predictions.numpy(),
    proposed_route.exit_indices.numpy(),
    runner.class_names,
    examples_path,
)
display(examples_figure)
plt.close(examples_figure)

## 15. Review exported artifacts

In [ ]:
artifact_rows = []
for category in ("manifests", "splits", "checkpoints", "cached_logits", "results", "figures", "logs"):
    folder = primary_artifact_root / category
    files = [path for path in folder.rglob("*") if path.is_file()]
    artifact_rows.append({"category": category, "files": len(files), "path": str(folder)})
artifact_table = pd.DataFrame(artifact_rows)
show_table("Artifact summary", artifact_table)

## 16. Complete the remaining full-run seeds

In [ ]:
seed_locked_stages = {primary_seed: locked_stage}
seed_profiling_stages = {primary_seed: profiling_stage}
if RUN_MODE == "full":
    for seed, seed_runner in runners.items():
        if seed == primary_seed:
            continue
        display(Markdown(f"# Running seed {seed}"))
        seed_outputs = run_seed_experiment(seed_runner, PROPOSED_VARIANT)
        seed_locked_stages[seed] = seed_outputs["locked_test"]
        seed_profiling_stages[seed] = seed_outputs["profiling"]
        show_stage(seed_outputs["locked_test"])
        show_stage(seed_outputs["ablations"], show_figures=False)
        show_stage(seed_outputs["profiling"], show_figures=False)

seed_raw, seed_summary = aggregate_seed_comparisons(seed_locked_stages)
aggregate_root = config.paths.artifact_root / "aggregate" / "results"
aggregate_root.mkdir(parents=True, exist_ok=True)
seed_raw.to_csv(aggregate_root / "locked_test_all_seeds.csv", index=False)
seed_summary.to_csv(aggregate_root / "locked_test_seed_summary.csv", index=False)
show_table("All seed results", seed_raw)
show_table("Mean, standard deviation, and 95% confidence interval", seed_summary)

latency_raw, latency_summary = aggregate_seed_tables(seed_profiling_stages, "latency", ("method",))
paired_raw, paired_summary = aggregate_seed_tables(seed_locked_stages, "paired_bootstrap", ("baseline",))
latency_raw.to_csv(aggregate_root / "latency_all_seeds.csv", index=False)
latency_summary.to_csv(aggregate_root / "latency_seed_summary.csv", index=False)
paired_raw.to_csv(aggregate_root / "paired_bootstrap_all_seeds.csv", index=False)
paired_summary.to_csv(aggregate_root / "paired_bootstrap_seed_summary.csv", index=False)
show_table("Latency across seeds", latency_raw)
show_table("Latency mean, standard deviation, and 95% confidence interval", latency_summary)
show_table("Paired Macro F1 bootstrap across seeds", paired_raw)
show_table("Paired Macro F1 difference summary", paired_summary)

assessment = assess_conclusion(config, runners, seed_locked_stages, seed_raw, seed_summary)
assessment_paths = assessment.save(aggregate_root)
show_table("Conclusion completeness checklist", assessment.checklist)
display(Markdown(f"## Conclusion status: `{assessment.status}`"))
for blocker in assessment.blockers:
    display(Markdown(f"- **Blocker:** {blocker}"))
for caveat in assessment.caveats:
    display(Markdown(f"- Caveat: {caveat}"))
display(Markdown(f"Checklist: `{assessment_paths['checklist']}`"))
display(Markdown(f"Machine-readable assessment: `{assessment_paths['assessment']}`"))

## 17. Takeaways from this executed run

In [ ]:
comparison = seed_summary.set_index("method")
proposed = comparison.loc["Proposed"]
final = comparison.loc["ResNet-18 Final-only"]
non_oracle = comparison.drop(index="Oracle", errors="ignore")
best_method = non_oracle["macro_f1_mean"].idxmax()
best = non_oracle.loc[best_method]
proposed_rank = int(non_oracle["macro_f1_mean"].rank(method="min", ascending=False).loc["Proposed"])
best_gap_pp = 100 * (proposed["macro_f1_mean"] - best["macro_f1_mean"])
flops_change = 100 * (1 - proposed["average_flops_mean"] / final["average_flops_mean"])
latency_by_method = latency_summary.set_index("method")
latency_ratio = (
    latency_by_method.loc["Proposed dynamic", "median_ms_mean"]
    / latency_by_method.loc["Full ResNet-18", "median_ms_mean"]
)
latency_word = "slower" if latency_ratio > 1 else "faster"

display(Markdown(f"# {assessment.status.title()}"))
display(Markdown(
    f"Proposed Macro F1 is **{proposed['macro_f1_mean']:.3f}**, ranking **{proposed_rank}/{len(non_oracle)}** "
    f"among non-oracle methods. The best method is **{best_method}** at **{best['macro_f1_mean']:.3f}**; "
    f"the Proposed gap is **{best_gap_pp:+.2f} percentage points**."
))
display(Markdown(
    f"Against final-only ResNet-18, estimated average FLOPs change by **{flops_change:+.1f}%**, but measured "
    f"{runners[primary_seed].device.type.upper()} batch-1 median latency is **{latency_ratio:.2f}x {latency_word}**. "
    "Therefore lower FLOPs alone are not reported as a real speed or energy gain."
))
if RUN_MODE == "smoke":
    display(Markdown("Run `full` with all three seeds before using these values in the report."))
progress.close("complete" if not assessment.blockers else "incomplete")
display(Markdown(f"Progress log closed: `{progress.aggregate_log_path}`"))